In [ ]:
# --- repo bootstrap -------------------------------------------------------
# Resolves the repository root and chdirs to it, so every path below is
# repo-relative and this notebook runs from any checkout location.
import os
from pathlib import Path

ROOT = Path.cwd().resolve()
while not (ROOT / ".git").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
os.chdir(ROOT)
print("repo root:", ROOT)


In [1]:
from ultralytics import YOLO

In [2]:
model_200_l = YOLO(model="experiments/person/yolo11l_ep220_stopped163/weights/best.pt")

In [3]:
#Run batched inference on a list of images
results = model_200_l.predict(source="data/raw/person/boot_images_sample2", imgsz=640, classes=[0,1], conf=0.5, save=True)  # return a list of Results objects



image 1/245 /home/ai_vison/Desktop/PPE/model_finetune/person/boot images sample 2/-01-15-1-1-1-4-18_jpg.rf.2k1JsxDtMbLsbnUFdLeM.jpg: 384x640 (no detections), 18.2ms
image 2/245 /home/ai_vison/Desktop/PPE/model_finetune/person/boot images sample 2/-01-15-1-1-1-4-33_jpg.rf.64vwk0zMg2eSDFfNBBiC.jpg: 384x640 1 half_person, 4.4ms
image 3/245 /home/ai_vison/Desktop/PPE/model_finetune/person/boot images sample 2/-01-15-1-3-1-3-30_jpg.rf.sO6mCMy8nSzT0KKRaetj.jpg: 384x640 1 half_person, 4.3ms
image 4/245 /home/ai_vison/Desktop/PPE/model_finetune/person/boot images sample 2/-01-16-2-1-2-2-315_jpg.rf.kij3ljMaoW5y6k4QVyga.jpg: 384x640 (no detections), 4.1ms
image 5/245 /home/ai_vison/Desktop/PPE/model_finetune/person/boot images sample 2/-01-16-2-1-2-2-317_jpg.rf.tmAQFHafGttj5YtwCLkz.jpg: 384x640 (no detections), 4.1ms
image 6/245 /home/ai_vison/Desktop/PPE/model_finetune/person/boot images sample 2/-01-16-2-1-2-2-36_jpg.rf.uIdIbRAjjgaDxIm7yMnP.jpg: 384x640 (no detections), 4.1ms
image 7/245 /hom

In [4]:
len(results)

245

In [5]:
import torchvision

def calculate_iou(box1, box2):
    # box_iou compares all boxes in box1 to all boxes in box2
    iou = torchvision.ops.box_iou(box1, box2)
    return iou

In [6]:
def is_overlap(box1, box2):
    iou = calculate_iou(box1, box2)
    print(iou)
    return iou > 0.7

def non_max_suppression(boxes):
    # Sort boxes by confidence in descending order
    try: 
        boxes = sorted(boxes, key=lambda x: x.conf, reverse=True)
        keep = []

        while boxes:
            # Take the box with highest confidence
            current = boxes.pop(0)
            keep.append(current)

            # Remove boxes that overlap too much with current box
            boxes = [box for box in boxes if
                        not is_overlap(current.xyxy, box.xyxy)]

        return keep
    except Exception as e:
        print(f"An error occurred during non-max suppression: {e}")
        return []

In [7]:
import os
import cv2

# Create a directory to save the extracted images
save_dir = "boot_data_2"
os.makedirs(save_dir, exist_ok=True)

def save_extracted_objects(keep, inf_img, img_index):
    for i, k in enumerate(keep):
        # Extract coordinates and convert them to integers
        x1, y1, x2, y2 = map(int, k.xyxy[0].tolist())
        
        # Crop from the original BGR image
        cropped_bgr = inf_img[0].orig_img[y1:y2, x1:x2]
        
        # Create a filename with the index and confidence score
        filename = os.path.join(save_dir, f"boot_2{img_index}_{i}.jpg")
        
        # Save the image to disk
        cv2.imwrite(filename, cropped_bgr)
        print(f"Saved: {filename}")

In [8]:
for j, inf_img in enumerate(results):
    keep = non_max_suppression(inf_img.boxes)
    save_extracted_objects(keep, inf_img, j)

Saved: boot_data_2/boot_21_0.jpg
Saved: boot_data_2/boot_22_0.jpg
Saved: boot_data_2/boot_26_0.jpg
Saved: boot_data_2/boot_210_0.jpg
Saved: boot_data_2/boot_211_0.jpg
tensor([[0.]], device='cuda:0')
Saved: boot_data_2/boot_212_0.jpg
Saved: boot_data_2/boot_212_1.jpg
Saved: boot_data_2/boot_214_0.jpg
tensor([[0.]], device='cuda:0')
Saved: boot_data_2/boot_215_0.jpg
Saved: boot_data_2/boot_215_1.jpg
tensor([[0.]], device='cuda:0')
Saved: boot_data_2/boot_216_0.jpg
Saved: boot_data_2/boot_216_1.jpg
Saved: boot_data_2/boot_217_0.jpg
Saved: boot_data_2/boot_218_0.jpg
Saved: boot_data_2/boot_219_0.jpg
tensor([[0.]], device='cuda:0')
tensor([[0.]], device='cuda:0')
tensor([[0.]], device='cuda:0')
tensor([[0.]], device='cuda:0')
tensor([[0.]], device='cuda:0')
tensor([[0.]], device='cuda:0')
tensor([[0.]], device='cuda:0')
tensor([[0.]], device='cuda:0')
tensor([[0.]], device='cuda:0')
tensor([[0.]], device='cuda:0')
tensor([[0.]], device='cuda:0')
tensor([[0.]], device='cuda:0')
tensor([[0.]]

In [9]:
# result = model_200_l.predict(source="data/raw/person/sample2/helmet_jacket_05081.jpg", imgsz=640, classes=[0,1], show=True, conf=0.5, save=True)  # return a list of Results objects
